In [0]:
%run ../config/config

In [0]:
# Imports
import sys
import time
import os
import pandas as pd
from pyspark.sql import SparkSession

sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger
from utils.control_available_info import control_available_info

In [0]:
# Logger check availability
logger = MLOpsLogger(
    name='02_check_availability',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': mes_vta,
        'environment': env,
        'notebook': '02_check_availability'
    }
)

In [0]:
try:
    logger.log_stage_start(
        'check_availability',
        mes_vta=mes_vta,
        environment=env
    )

    start_time = time.time()
    spark = SparkSession.builder.getOrCreate()

    # =============================================================================
    # 1. VERIFICAR DISPONIBILIDAD VIA ARCHIVO DE CONTROL (si existe)
    # =============================================================================
    logger.info("=" * 60)
    logger.info("VERIFICANDO DISPONIBILIDAD DE TABLAS FUENTE")
    logger.info("=" * 60)

    num_errors = 0

    mapping_file = os.path.join(mapping_path, "available_info.txt")
    if os.path.exists(mapping_file):
        logger.info(f"Usando archivo de control: {mapping_file}")
        reporte_disp, num_errors = control_available_info(
            in_file_path=mapping_file,
            in_month_ref=str(mes_vta)
        )
        logger.info(f"Revisión de archivo de control: {num_errors} errores")
    else:
        logger.info("Archivo available_info.txt no encontrado. Verificando tablas directamente...")
        reporte_disp = pd.DataFrame()

    # =============================================================================
    # 4. RESUMEN Y TASK VALUES
    # =============================================================================
    duration = time.time() - start_time

except Exception as e:
    logger.log_exception(
        operation='check_availability',
        exception=e,
        should_raise=True,
        mes_vta=mes_vta,
        environment=env
    )

finally:
    if 'logger' in locals():
        logger.info(f"Finalizando notebook: {logger.name}")
        logger._flush_all_handlers()
        logger.close()